# L12d Lab: Exploration and Reproducibility in Q-Learning
In this lab, we use Q-learning to learn how to feed a bioreactor, and we look at two things that can go wrong: a result that changes with the random seed, and an agent that never learns because it never tries anything new.

__Background:__ A __bioreactor__ is a tank in which living cells make a product, here an antibody, a protein used as a drug. The cells grow on sugar that we feed into the tank. In a __fed-batch__ run, we start with a half-full tank and add feed each cycle until the tank is full, and then we __harvest__ it: we collect the contents and the product the cells made. Cells that get more sugar than they can use turn the excess into __lactate__, a waste product that slows their growth. If the lactate builds up too far, the culture __crashes__ and the batch is lost.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Formulate feeding a bioreactor as a Markov decision process:__ The state is the amount of cells, the amount of lactate, and the volume of liquid in the tank, and each action is a feed rate that changes all three. We'll compute the optimal feeding policy with value iteration to have a benchmark.
> * __Make a Q-learning result reproducible:__ Q-learning makes random choices, so two training runs give the same table only if they use the same seed for the random number generator. We'll see how much a short training run depends on its seed, and how longer training removes most of that dependence.
> * __Diagnose an agent that does not explore:__ A greedy agent that never tries a new action can stay stuck on its first choice forever. We'll find the cause in the visit counts and fix it with exploration.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) for the Q-learning agent and the policy. Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The bioreactor model, value iteration, Q-learning, and the batch simulator are in [src/Bioreactor.jl](src/Bioreactor.jl), and the batch figure is in [src/Visualize.jl](src/Visualize.jl). This code is complete; there is nothing to edit in this lab.
___

## Task 1: Build the Bioreactor and Its Optimal Policy
In this task, we build the bioreactor model, try four fixed feed rates, and compute the optimal feeding policy with value iteration.

We describe the culture with three variables, each scaled to lie between 0 and 1: the __cell density__ $X$, the amount of cells in each unit of liquid; the __lactate concentration__ $L$, the amount of lactate in each unit of liquid; and the volume $V$ of liquid in the tank.

Each cycle $t$, we choose a feed rate $f_{t}\in\{0,\tfrac{1}{3},\tfrac{2}{3},1\}$. The feed adds the volume $v_{f}f_{t}$, which __dilutes__ the culture: the same cells and lactate now fill more liquid, so each concentration falls by the fraction $D_{t}=v_{f}f_{t}/V_{t}$. The culture changes by:

$$
\begin{align*}
X_{t+1} &= X_{t} + \underbrace{\mu_{t}\,X_{t}(1 - X_{t})}_{\text{growth}} - \underbrace{k_{d}\,X_{t}}_{\text{death}} - \underbrace{D_{t}\,X_{t}}_{\text{dilution}}\\
L_{t+1} &= L_{t} + \underbrace{y_{L}\,f_{t}\,X_{t}}_{\text{lactate made}} - \underbrace{k_{L}\,L_{t}}_{\text{consumed}} - \underbrace{D_{t}\,L_{t}}_{\text{dilution}}\\
V_{t+1} &= V_{t} + v_{f}\,f_{t}
\end{align*}
$$

The growth rate $\mu_{t}$ rises with the feed and falls as lactate builds up:

$$
\begin{equation*}
\mu_{t} = \mu_{\max}\cdot\frac{f_{t}}{K_{f}+f_{t}}\cdot\frac{K_{I}}{K_{I}+L_{t}}
\end{equation*}
$$

where $\mu_{\max}$ is the largest growth rate, $K_{f}$ is the feed rate at which the first fraction, the feed factor, equals one half, and $K_{I}$ is the lactate level at which the second fraction, the lactate factor, equals one half. The other constants are the fraction of the cells that die each cycle, $k_{d}$, the lactate made per unit of feed and cell density, $y_{L}$, the fraction of the lactate the cells consume each cycle, $k_{L}$, and the volume added per cycle at full feed, $v_{f}$.

We round each new value to the nearest of 21 levels, 0.05 apart, so there are $21^{3}=9261$ states. The rounding changes how the feeds fill the tank. The full feed adds 0.05, one level, each cycle. The low feed adds a third of a level, which rounds back to the same volume, and the medium feed adds two thirds of a level, which rounds up to a full level.

Two kinds of states are absorbing, and the batch ends when the culture enters one:
* __Harvest:__ the tank is full, $V\geq V_{\max}$. The reward is the product harvested, $q_{P}\,X\,V$, where $q_{P}$ is the product made per unit of cell density and volume.
* __Crash:__ the lactate reaches $L\geq L_{\text{crash}}$, which earns the reward $R_{\text{crash}}<0$. A state that meets both conditions counts as a crash.

Every other cycle earns nothing: we are paid only at the end, for what we harvest.

We encode the bioreactor with [the `MyFedBatchBioreactorModel` type](src/Bioreactor.jl), and [the `build(...)` function](src/Bioreactor.jl) takes the parameters in a `NamedTuple`. Let's build the bioreactor:

In [ ]:
env = build(MyFedBatchBioreactorModel, (
    nlevels = 21, # 21 levels on each axis: X, L, and V take the values 0, 0.05, ..., 1
    feeds = [0.0, 1/3, 2/3, 1.0], # the feed rates of the four actions: none, low, medium, high
    mumax = 1.00, # largest growth rate per cycle
    Kf = 0.30, # feed rate at which the feed factor f/(Kf + f) is one half
    KIL = 0.35, # lactate level at which the lactate factor KIL/(KIL + L) is one half
    kd = 0.03, # fraction of the cells that die per cycle
    yL = 1.00, # lactate made per unit of feed and cell density
    kL = 0.30, # fraction of the lactate consumed per cycle
    qP = 1.0, # product per unit of cell density and volume: a harvest earns qP⋅X⋅V
    vfeed = 0.05, # volume added per cycle at full feed
    V0 = 0.50, # starting volume: the tank is half full
    Vmax = 1.00, # harvest when V ≥ 1, a full tank
    Lcrash = 0.90, # the culture crashes when L ≥ 0.90
    Rcrash = -25.0, # reward for a crash
));
nstates = env.nlevels^3; # the number of states, 21 × 21 × 21
nactions = length(env.feeds); # the number of actions, one per feed rate
γ = 0.95; # discount factor
s₀ = stateindex(env, 0.10, 0.0, env.V0); # every batch starts with a few cells, no lactate, and a half-full tank

### Fixed feed rates
[The `simulate(...)` function](src/Bioreactor.jl) runs a policy from a starting state until the batch is harvested, the culture crashes, or 80 cycles pass, a timeout. It returns the product harvested and the discounted return $G=r_{1}+\gamma r_{2}+\gamma^{2}r_{3}+\cdots$, the quantity the optimal policy makes as large as possible.

__Predict:__ Which of the four fixed feed rates, a policy that feeds at the same rate in every state, will harvest a batch?

In [ ]:
let
    df = DataFrame(); # one row per fixed feed rate
    labels = ["none", "low", "medium", "high"];
    for a ∈ 1:nactions
        π_fixed = fill(a, nstates); # the same action a in every state
        batch = simulate(env, π_fixed, s₀; γ = γ);
        push!(df, (feed = labels[a], outcome = String(batch.outcome), cycles = batch.steps,
            product = round(batch.product, digits = 2)));
    end
    pretty_table(df; backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

__What do we see?__ No fixed feed rate harvests a batch. With no feed, the tank never fills. With the low feed, it never fills on our grid either, because each cycle's volume rounds back to the same level. With the medium or high feed, the lactate crashes the culture first.

### The optimal policy
We know the dynamics, so value iteration gives the optimal policy. [The `solve(...)` function](src/Bioreactor.jl) for the bioreactor runs value iteration and returns the optimal action values $Q^{*}(s,a)$, and [the `policy(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.policy-Tuple{Matrix{Float64}}) reads the best action from each row. [The `plot_batch(...)` function](src/Visualize.jl) draws a batch over time:

In [ ]:
Qstar = solve(env; γ = γ); # value iteration: the optimal action values
π_star = policy(Qstar); # the optimal policy
optimal = simulate(env, π_star, s₀; γ = γ); # the optimal batch
println("optimal policy: ", optimal.outcome, " after ", optimal.steps, " cycles, product ",
    round(optimal.product, digits = 2), ", discounted return G = ", round(optimal.G, digits = 3))
plot_batch(env, optimal; title = "Optimal policy (value iteration)")

__What do we see?__ The optimal policy feeds at the medium rate in most cycles, which grows the cells and fills the tank. In the last few cycles, it alternates feeding with rest cycles of no feed, which let the lactate fall back from near the crash level before the next feed. Its discounted return is the benchmark for every policy we learn below.

___

## Task 2: Reproducibility
In this task, we train a Q-learning agent twice with the same seed and once with another seed, and compare the batches of the learned policies.

Q-learning makes random choices: the starting state of each episode, and the exploring actions. [The `solve(...)` function](src/Bioreactor.jl) for a `MyQLearningAgentModel` agent and the bioreactor draws these from a random number generator started from a __seed__, a whole number we choose. The same seed gives the same sequence of random numbers, and so the same training run.

> __Training:__ The learning rate of the $n$-th update of a pair $(s,a)$ is $1/n$. The exploration probability $\epsilon$ starts at `ϵ_start` and falls by the same amount each episode until it reaches `ϵ_floor`. With the defaults, 1 and 0.05, it falls from 1 to 0.05 over the first 70 percent of the episodes. Each episode starts in a random state and runs for at most 80 cycles.

We build the agent with [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/), which takes the states, the actions, a learning rate, the discount factor, and the starting table, all zeros. Let's train it for 20,000 episodes with the seed 5800:

In [ ]:
agent = build(MyQLearningAgentModel, (
    states = collect(1:nstates), # the states
    actions = collect(1:nactions), # the actions
    α = 0.1, # learning rate field (solve(...) uses 1/n instead)
    γ = γ, # the discount factor
    Q = zeros(nstates, nactions), # the starting table: all zeros
));
result = solve(agent, env; episodes = 20_000, seed = 5800); # Q-learning: returns the table Q and the visit counts
learned = simulate(env, policy(result.Q), s₀; γ = γ); # the batch of the learned policy
println("seed 5800: ", learned.outcome, " after ", learned.steps, " cycles, product ",
    round(learned.product, digits = 2), ", G = ", round(learned.G, digits = 3))

Let's train again with the same seed, and check that we get exactly the same table:

In [ ]:
result_again = solve(agent, env; episodes = 20_000, seed = 5800); # the same training run
result_again.Q == result.Q # true: every entry is the same

__Predict:__ Suppose we train with the seed 1 instead. Will the agent learn the same policy, and harvest the same batch?

In [ ]:
let
    # your turn: change the training and rerun (try other values) -
    seed_new = 1; # the seed of the random number generator
    episodes_new = 20_000; # the number of episodes

    # train, follow the learned policy, and draw the batch -
    result_new = solve(agent, env; episodes = episodes_new, seed = seed_new);
    batch_new = simulate(env, policy(result_new.Q), s₀; γ = γ);
    println("seed ", seed_new, ", ", episodes_new, " episodes: ", batch_new.outcome, " after ", batch_new.steps,
        " cycles, product ", round(batch_new.product, digits = 2), ", G = ", round(batch_new.G, digits = 3))
    plot_batch(env, batch_new; title = "Learned policy, seed $(seed_new)") # return
end

__What do we see?__ A different seed gave a different batch: a different number of cycles and a different product. Both discounted returns are below the optimal one. After 20,000 episodes, the agent has tried many state-action pairs only a few times or not at all, so which policy it learns depends on the luck of the draws.

### Things to think about

__Question:__ Set `episodes_new = 200_000` and rerun the cell with `seed_new` equal to 5800, 1, and 2. Do the three batches agree now? How close is each discounted return to the optimal one? Why does more training make the result depend less on the seed? Briefly explain.

___

## Task 3: Exploration and Diagnosis
In this task, we train an agent that never explores, find out from its visit counts why it fails, and compare it with an agent that explores.

Every reward before the end of a batch is zero, and the table starts at zero. A __greedy__ agent never explores: it always takes the action with the largest value, the first one when several tie.

__Predict:__ A greedy agent, with `ϵ_start = 0.0` and `ϵ_floor = 0.0`, trains for 200,000 episodes. Will its policy harvest a batch?

In [ ]:
greedy = solve(agent, env; episodes = 200_000, seed = 5800, ϵ_start = 0.0, ϵ_floor = 0.0); # never explores
greedy_batch = simulate(env, policy(greedy.Q), s₀; γ = γ);
println("greedy agent: ", greedy_batch.outcome, " after ", greedy_batch.steps, " cycles, product ",
    round(greedy_batch.product, digits = 2))

__What do we see?__ The greedy agent never harvests. Training ran without an error, so nothing told us that it failed. To find the cause, let's look at the visit counts: how many times the agent took each action during training. [The `sum(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.sum) with `dims = 1` adds up each column of `visits`, one column per action:

In [ ]:
sum(greedy.visits, dims = 1) # the number of times the greedy agent took each feed: none, low, medium, high

__What do we see?__ The greedy agent only ever chose "no feed". All four actions start at zero, so the first one, "no feed", wins the tie. Feeding nothing earns zero, so its value stays zero and the tie never breaks. With no feed, the tank never fills, so the agent never reaches a harvest and never sees a reward that could change its mind.

[The `count_untried_pairs(...)` function](src/Bioreactor.jl) counts the state-action pairs an agent never tried, in the states that are not absorbing:

In [ ]:
count_untried_pairs(env, greedy.visits) # (pairs never tried, pairs checked)

__Predict:__ Now let's train an agent that explores, with the default exploration that falls from 1 to 0.05. Will it harvest, and how many pairs will it leave untried?

In [ ]:
let
    # your turn: change the training and rerun (try other values) -
    episodes_new = 200_000; # the number of episodes
    ϵ_start_new = 1.0; # the exploration probability at the start
    ϵ_floor_new = 0.05; # the smallest exploration probability

    # train, count the untried pairs, and follow the learned policy -
    result_new = solve(agent, env; episodes = episodes_new, seed = 5800,
        ϵ_start = ϵ_start_new, ϵ_floor = ϵ_floor_new);
    (number_untried, number_checked) = count_untried_pairs(env, result_new.visits);
    batch_new = simulate(env, policy(result_new.Q), s₀; γ = γ);
    println(episodes_new, " episodes, ϵ from ", ϵ_start_new, " to ", ϵ_floor_new, ": ", number_untried, " of ",
        number_checked, " pairs never tried")
    println("visits of each feed: ", sum(result_new.visits, dims = 1))
    println(batch_new.outcome, " after ", batch_new.steps, " cycles, product ",
        round(batch_new.product, digits = 2), ", G = ", round(batch_new.G, digits = 3))
    plot_batch(env, batch_new; title = "Learned policy, exploring agent") # return
end

__What do we see?__ The exploring agent tried every feed rate many times and left only a few pairs untried, and its policy harvests a batch with the optimal product. Exploration let it find the harvest, and the reward of the harvest then spread back to the earlier states.

### Things to think about

__Question:__ Set `episodes_new = 2_000` and rerun the cell. The agent explores, but does it harvest, and how many pairs did it leave untried? Then set `episodes_new = 200_000` with `ϵ_start_new = 0.1` and `ϵ_floor_new = 0.1`, a fixed exploration probability. Does this agent harvest, and how many pairs did it leave untried? What does each run tell us about how much experience a table of this size needs? Briefly explain.

___

## Summary
In this lab, we computed the optimal feeding policy of a bioreactor with value iteration, and used it to judge Q-learning runs with different seeds and different amounts of exploration.

> __Key Takeaways:__
>
> * __Value iteration gives the benchmark:__ With the model known, value iteration found a feeding policy that harvests the batch, where on our grid every fixed feed rate either never fills the tank or crashes the culture. We used its discounted return to judge every learned policy.
>
> * __A seed makes a run repeatable, not right:__ The same seed reproduced the same table exactly, but a short training run learned a different policy for each seed. Longer training brought the policies from different seeds close to the optimum.
>
> * __An agent that never explores can fail silently:__ With zero rewards until the harvest and a table of zeros, a greedy agent never fed the culture and never saw a harvest. The visit counts showed the cause, and exploration fixed it.

Before we trust a learned policy, we compare it with a benchmark, rerun it with other seeds, and check which actions the agent actually tried.

___